In [1]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
!pip install mordred --no-index --find-links=file:///kaggle/input/mordred-1-2-0-py3-none-any/

Processing /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
Looking in links: file:///kaggle/input/mordred-1-2-0-py3-none-any/
Processing /kaggle/input/mordred-1-2-0-py3-none-any/mordred-1.2.0-py3-none-any.whl
Processing /kaggle/input/mordred-1-2-0-py3-none-any/networkx-2.8.8-py3-none-any.whl (from mordred)
  Attempting uninstall: networkx
    Found existing installation: networkx 3.5
    Uninstalling networkx-3.5:
      Successfully uninstalled networkx-3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import re
import requests
import joblib, numpy

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import scipy.stats as stats

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [3]:
# Load data
train = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')
test = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

def improved_tokenize_smiles(smiles):
    """
    Improved SMILES tokenization with better pattern matching
    """
    # More comprehensive pattern for SMILES tokens
    pattern = r'(\[[^\[\]]{1,10}\]|Br|Cl|[bcnops]|[BCNOFPSI]|[()=+\-#@:/\\]|\d+|\.)'
    tokens = re.findall(pattern, smiles)
    
    # Handle remaining characters not caught by pattern
    reconstructed = ''.join(tokens)
    remaining = smiles.replace(reconstructed, '')
    if remaining:
        tokens.extend(list(remaining))
    
    return tokens

def create_smiles_features(train_smiles, test_smiles):
    """
    Create comprehensive SMILES-based features
    """
    # Tokenize all SMILES
    train_tokens = [improved_tokenize_smiles(smiles) for smiles in train_smiles]
    test_tokens = [improved_tokenize_smiles(smiles) for smiles in test_smiles]
    
    # Create vocabulary from all tokens
    all_tokens = []
    for tokens in train_tokens + test_tokens:
        all_tokens.extend(tokens)
    
    vocab = sorted(list(set(all_tokens)))
    vocab_dict = {token: idx for idx, token in enumerate(vocab)}
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Max sequence length: {max([len(t) for t in train_tokens + test_tokens])}")
    
    # Method 1: Token counts (bag of tokens)
    def get_token_counts(token_list, vocab_dict):
        features = np.zeros(len(vocab_dict))
        for tokens in token_list:
            for token in tokens:
                if token in vocab_dict:
                    features[vocab_dict[token]] += 1
        return features
    
    train_counts = np.array([get_token_counts([tokens], vocab_dict) for tokens in train_tokens])
    test_counts = np.array([get_token_counts([tokens], vocab_dict) for tokens in test_tokens])
    
    # Method 2: Sequence features (padded sequences)
    max_len = max([len(t) for t in train_tokens + test_tokens])
    max_len = min(max_len, 500)  # Cap at 500 to avoid memory issues
    
    def pad_sequence(tokens, max_len, vocab_dict):
        sequence = []
        for i, token in enumerate(tokens[:max_len]):
            if token in vocab_dict:
                sequence.append(vocab_dict[token])
            else:
                sequence.append(0)  # Unknown token
        
        # Pad with zeros
        while len(sequence) < max_len:
            sequence.append(0)
        
        return sequence
    
    train_sequences = np.array([pad_sequence(tokens, max_len, vocab_dict) for tokens in train_tokens])
    test_sequences = np.array([pad_sequence(tokens, max_len, vocab_dict) for tokens in test_tokens])
    
    # Method 3: Chemical descriptors from SMILES
    def get_chemical_descriptors(smiles):
        """
        Extract chemical descriptors from SMILES string
        """
        features = {}
        
        # Basic molecular features
        features['length'] = len(smiles)
        features['num_atoms'] = len(re.findall(r'[A-Z][a-z]?', smiles))
        features['num_bonds'] = smiles.count('=') + smiles.count('#')
        features['num_rings'] = smiles.count('1') + smiles.count('2') + smiles.count('3') + \
                               smiles.count('4') + smiles.count('5') + smiles.count('6')
        features['num_branches'] = smiles.count('(') + smiles.count(')')
        features['num_aromatic'] = smiles.count('[') + smiles.count(']')
        
        # Atom counts
        features['carbon_count'] = smiles.count('C') + smiles.count('c')
        features['nitrogen_count'] = smiles.count('N') + smiles.count('n')
        features['oxygen_count'] = smiles.count('O') + smiles.count('o')
        features['sulfur_count'] = smiles.count('S') + smiles.count('s')
        features['phosphorus_count'] = smiles.count('P') + smiles.count('p')
        features['fluorine_count'] = smiles.count('F')
        features['chlorine_count'] = smiles.count('Cl')
        features['bromine_count'] = smiles.count('Br')
        
        # Functional groups
        features['alcohol_groups'] = smiles.count('OH')
        features['carboxyl_groups'] = smiles.count('COOH')
        features['ester_groups'] = smiles.count('COO')
        features['ether_groups'] = smiles.count('O') - features['alcohol_groups'] - \
                                  features['carboxyl_groups'] - features['ester_groups']
        
        # Structural complexity
        features['complexity_score'] = features['num_rings'] + features['num_branches'] + \
                                     features['num_bonds'] * 0.5
        
        # Ratios
        total_atoms = max(features['num_atoms'], 1)
        features['heteroatom_ratio'] = (features['nitrogen_count'] + features['oxygen_count'] + 
                                      features['sulfur_count'] + features['phosphorus_count']) / total_atoms
        features['carbon_ratio'] = features['carbon_count'] / total_atoms
        
        return features
    
    # Extract chemical descriptors
    train_descriptors = [get_chemical_descriptors(smiles) for smiles in train_smiles]
    test_descriptors = [get_chemical_descriptors(smiles) for smiles in test_smiles]
    
    descriptor_names = list(train_descriptors[0].keys())
    
    train_desc_df = pd.DataFrame(train_descriptors, columns=descriptor_names)
    test_desc_df = pd.DataFrame(test_descriptors, columns=descriptor_names)
    
    return {
        'vocab': vocab_dict,
        'train_counts': train_counts,
        'test_counts': test_counts,
        'train_sequences': train_sequences,
        'test_sequences': test_sequences,
        'train_descriptors': train_desc_df,
        'test_descriptors': test_desc_df
    }

In [4]:
smiles_features = create_smiles_features(train['SMILES'], test['SMILES'])

Vocabulary size: 100
Max sequence length: 610


In [5]:
smiles_features['train_descriptors']

,length,num_atoms,num_bonds,num_rings,num_branches,num_aromatic,carbon_count,nitrogen_count,oxygen_count,sulfur_count,...,fluorine_count,chlorine_count,bromine_count,alcohol_groups,carboxyl_groups,ester_groups,ether_groups,complexity_score,heteroatom_ratio,carbon_ratio
0,26,11,1,2,4,0,15,0,2,0,...,0,0,0,0,0,0,2,6.5,0.181818,1.363636
1,82,23,0,10,16,4,43,2,0,0,...,0,0,0,0,0,0,0,26.0,0.086957,1.869565
2,134,25,7,14,28,0,62,0,9,2,...,0,0,0,0,0,0,9,45.5,0.440000,2.480000
3,79,6,0,12,18,0,40,2,0,0,...,0,0,0,0,0,0,0,30.0,0.333333,6.666667
4,118,46,4,12,18,8,54,4,12,0,...,0,0,0,0,0,0,12,32.0,0.347826,1.173913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7968,44,14,2,4,10,0,22,0,4,0,...,0,0,0,0,0,0,4,15.0,0.285714,1.571429
7969,110,26,8,14,24,4,40,6,10,0,...,0,0,0,0,0,0,10,42.0,0.615385,1.538462
7970,73,21,5,10,16,0,31,3,5,0,...,0,0,0,0,0,0,5,28.5,0.380952,1.476190
7971,16,3,1,2,2,0,9,0,0,0,...,0,0,0,0,0,0,0,4.5,0.000000,3.000000


In [6]:
smiles_features['test_descriptors']

,length,num_atoms,num_bonds,num_rings,num_branches,num_aromatic,carbon_count,nitrogen_count,oxygen_count,sulfur_count,...,fluorine_count,chlorine_count,bromine_count,alcohol_groups,carboxyl_groups,ester_groups,ether_groups,complexity_score,heteroatom_ratio,carbon_ratio
0,71,15,2,8,20,0,29,2,2,0,...,6,0,0,0,0,0,2,29.0,0.266667,1.933333
1,71,9,2,10,18,0,35,0,4,0,...,0,0,0,0,0,0,4,29.0,0.444444,3.888889
2,75,20,4,12,12,0,36,2,6,0,...,0,0,0,0,0,0,6,26.0,0.400000,1.800000


In [7]:
def get_ngram_features(tokens, top_ngrams, n=2):
    features = np.zeros(len(top_ngrams))
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        if ngram in top_ngrams:
            features[top_ngrams.index(ngram)] += 1
    return features

def create_ngram_features(token_list, n=2, top_k=1000):
    """
    Create n-gram features from tokenized SMILES
    """
    from collections import Counter
    
    # Generate n-grams
    all_ngrams = []
    for tokens in token_list:
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i:i+n])
            all_ngrams.append(ngram)
    
    # Get most common n-grams
    ngram_counts = Counter(all_ngrams)
    top_ngrams = [ngram for ngram, _ in ngram_counts.most_common(top_k)]
    
    # Create features    
    def get_ngram_features(tokens, top_ngrams):
        features = np.zeros(len(top_ngrams))
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i:i+n])
            if ngram in top_ngrams:
                features[top_ngrams.index(ngram)] += 1
        return features
    
    return top_ngrams, [get_ngram_features(tokens, top_ngrams) for tokens in token_list]

In [8]:
# Retokenize for n-grams
train_tokens_list = [improved_tokenize_smiles(smiles) for smiles in train['SMILES']]
test_tokens_list = [improved_tokenize_smiles(smiles) for smiles in test['SMILES']]

# Create bigram and trigram features
top_bigrams, train_bigram_features = create_ngram_features(train_tokens_list + test_tokens_list, n=2, top_k=500)
_, test_bigram_features = create_ngram_features(test_tokens_list, n=2, top_k=500)
test_bigram_features = [get_ngram_features(tokens, top_bigrams) for tokens in test_tokens_list]

train_bigram_features = [get_ngram_features(tokens, top_bigrams, 2) for tokens in train_tokens_list]
test_bigram_features = [get_ngram_features(tokens, top_bigrams, 2) for tokens in test_tokens_list]

# Convert to DataFrames
train_bigram_df = pd.DataFrame(train_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])
test_bigram_df = pd.DataFrame(test_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])

# Combine all features
print("Combining features...")

Combining features...


In [9]:
train_base = train.drop(columns=['SMILES'] if 'SMILES' in train.columns else [])
test_base = test.drop(columns=['SMILES'] if 'SMILES' in test.columns else [])

# Add token count features (most informative)
train_counts_df = pd.DataFrame(smiles_features['train_counts'], 
                              columns=[f'token_count_{i}' for i in range(smiles_features['train_counts'].shape[1])])
test_counts_df = pd.DataFrame(smiles_features['test_counts'], 
                             columns=[f'token_count_{i}' for i in range(smiles_features['test_counts'].shape[1])])

# Convert bigram features to DataFrames
train_bigram_df = pd.DataFrame(train_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])
test_bigram_df = pd.DataFrame(test_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])

# Add chemical descriptors
train_final = pd.concat([
    train_base,
    smiles_features['train_descriptors'],
], axis=1)

test_final = pd.concat([
    test_base,
    smiles_features['test_descriptors'], 
], axis=1)

print(f"Final feature count: {train_final.shape[1]}")

Final feature count: 27


In [10]:
train_final

,id,Tg,FFV,Tc,Density,Rg,length,num_atoms,num_bonds,num_rings,...,fluorine_count,chlorine_count,bromine_count,alcohol_groups,carboxyl_groups,ester_groups,ether_groups,complexity_score,heteroatom_ratio,carbon_ratio
0,87817,NaN,0.374645,0.205667,NaN,NaN,26,11,1,2,...,0,0,0,0,0,0,2,6.5,0.181818,1.363636
1,106919,NaN,0.370410,NaN,NaN,NaN,82,23,0,10,...,0,0,0,0,0,0,0,26.0,0.086957,1.869565
2,388772,NaN,0.378860,NaN,NaN,NaN,134,25,7,14,...,0,0,0,0,0,0,9,45.5,0.440000,2.480000
3,519416,NaN,0.387324,NaN,NaN,NaN,79,6,0,12,...,0,0,0,0,0,0,0,30.0,0.333333,6.666667
4,539187,NaN,0.355470,NaN,NaN,NaN,118,46,4,12,...,0,0,0,0,0,0,12,32.0,0.347826,1.173913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7968,2146592435,NaN,0.367498,NaN,NaN,NaN,44,14,2,4,...,0,0,0,0,0,0,4,15.0,0.285714,1.571429
7969,2146810552,NaN,0.353280,NaN,NaN,NaN,110,26,8,14,...,0,0,0,0,0,0,10,42.0,0.615385,1.538462
7970,2147191531,NaN,0.369411,NaN,NaN,NaN,73,21,5,10,...,0,0,0,0,0,0,5,28.5,0.380952,1.476190
7971,2147435020,261.662355,NaN,NaN,NaN,NaN,16,3,1,2,...,0,0,0,0,0,0,0,4.5,0.000000,3.000000


In [11]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

for i in zip(train_final.columns, train_final.dtypes):
    if i[0] != 'id':
        if i[1]=='O':
            encoder.fit(train_final[i[0]])
            
            train_final[i[0]] = train_final[i[0]].fillna('Unknown')
    
            test_final[i[0]] = test_final[i[0]].fillna('Unknown')
        else:
            normalize.fit(numpy.array(train_final[i[0]]).reshape(-1,1))
            
            train_final[i[0]].fillna(0, inplace=True)
            
            if i[0] in test_final.columns:
                test_final[i[0]].fillna(0, inplace=True)


In [12]:
# Improved preprocessing
def improved_preprocessing(train_df, test_df, target_cols):
    """
    Improved preprocessing with proper encoding and scaling
    """
    # Separate features and targets
    feature_cols = [col for col in train_df.columns if col not in ['id'] + target_cols]
    
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df[target_cols].copy()
    
    # Handle categorical features
    categorical_cols = X_train.select_dtypes(include=['object']).columns
    
    for col in categorical_cols:
        # Fill missing values
        X_train[col] = X_train[col].fillna('Unknown')
        X_test[col] = X_test[col].fillna('Unknown')
        
        # Label encoding
        encoder = LabelEncoder()
        combined_data = pd.concat([X_train[col], X_test[col]])
        encoder.fit(combined_data)
        
        X_train[col] = encoder.transform(X_train[col])
        X_test[col] = encoder.transform(X_test[col])
    
    # Handle numerical features
    numerical_cols = X_train.select_dtypes(include=[np.number]).columns
    
    # Impute missing values
    imputer = SimpleImputer(strategy='median')
    X_train[numerical_cols] = imputer.fit_transform(X_train[numerical_cols])
    X_test[numerical_cols] = imputer.transform(X_test[numerical_cols])
    
    # Scale features (important for neural networks and some models)
    scaler = StandardScaler()
    X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
    X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])
    
    # Feature selection to reduce dimensionality
    print("Performing feature selection...")
    
    # Use RandomForest for feature importance
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train.iloc[:, 0])  # Use first target for feature selection
    
    # Get feature importance
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Select top features
    top_features = feature_importance.head(2000)['feature'].tolist()  # Adjust number as needed
    
    X_train_selected = X_train[top_features]
    X_test_selected = X_test[top_features]
    
    print(f"Selected {len(top_features)} most important features")
    
    return X_train_selected, X_test_selected, y_train, feature_importance

# Apply improved preprocessing
target_columns = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']  # Adjust based on your targets

X_train_processed, X_test_processed, y_train_processed, feature_importance = improved_preprocessing(
    train_final, test_final, target_columns
)

# Split for validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_processed, y_train_processed, test_size=0.2, random_state=42
)

y_train_split_Tg = y_train_split['Tg']
y_train_split_FFV = y_train_split['FFV']
y_train_split_Tc = y_train_split['Tc']
y_train_split_Density = y_train_split['Density']
y_train_split_Rg = y_train_split['Rg']

y_val_split_Tg = y_val_split['Tg']
y_val_split_FFV = y_val_split['FFV']
y_val_split_Tc = y_val_split['Tc']
y_val_split_Density = y_val_split['Density']
y_val_split_Rg = y_val_split['Rg']

Performing feature selection...
Selected 21 most important features


In [13]:
X_train_split

,carbon_ratio,length,heteroatom_ratio,complexity_score,num_atoms,carbon_count,num_bonds,num_rings,num_branches,nitrogen_count,...,ether_groups,num_aromatic,sulfur_count,fluorine_count,chlorine_count,bromine_count,phosphorus_count,carboxyl_groups,ester_groups,alcohol_groups
2404,-0.368435,0.468501,-0.203453,-0.143753,1.958927,0.853164,-0.129134,-0.167952,-0.104177,-0.991914,...,2.916078,-0.275625,-0.349186,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
1514,-0.316151,-0.399832,-0.342285,-0.579454,0.242287,-0.138210,-0.129134,-0.542153,-0.544699,0.210356,...,-0.504231,-0.275625,1.038681,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
265,-0.062729,1.486547,-0.193898,1.816897,0.456867,1.447989,0.820228,1.703056,1.657910,0.210356,...,0.521862,-0.275625,-0.349186,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
0,-0.327094,-0.968740,-0.566552,-1.051462,-0.401453,-0.733034,-0.603814,-0.916355,-0.985221,-0.991914,...,-0.504231,-0.275625,-0.349186,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
1451,-0.182400,-0.130349,0.341196,0.037788,-0.401453,-0.270393,0.345547,0.206250,-0.104177,0.210356,...,0.179831,-0.275625,1.038681,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5226,-0.030816,-1.238223,-0.263969,-1.087770,-1.259773,-1.195675,-1.078495,-0.916355,-0.985221,-0.991914,...,-0.846262,-0.275625,-0.349186,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
5390,-0.561360,-0.130349,-0.929651,0.219330,-0.294163,-1.460042,-1.078495,0.206250,0.336345,-0.991914,...,-1.188293,9.341689,-0.349186,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
860,-0.267666,-0.968740,-0.680020,-0.797304,-0.723323,-0.865217,-1.078495,-0.916355,-0.544699,-0.991914,...,-0.846262,-0.275625,-0.349186,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0
7603,-0.000499,0.079248,0.268576,0.183022,-0.508743,0.126157,0.345547,0.206250,0.116084,0.210356,...,0.179831,-0.275625,-0.349186,-0.27968,-0.125371,-0.090805,-0.16041,0.0,0.0,0.0


In [14]:
y_train_split

,Tg,FFV,Tc,Density,Rg
2404,0.00000,0.347705,0.000000,0.0,0.0
1514,0.00000,0.376183,0.000000,0.0,0.0
265,0.00000,0.409783,0.000000,0.0,0.0
0,0.00000,0.374645,0.205667,0.0,0.0
1451,0.00000,0.365494,0.000000,0.0,0.0
...,...,...,...,...,...
5226,0.00000,0.341451,0.000000,0.0,0.0
5390,85.26711,0.000000,0.000000,0.0,0.0
860,0.00000,0.374923,0.000000,0.0,0.0
7603,0.00000,0.353787,0.000000,0.0,0.0


In [15]:
# Improved CatBoost parameters
improved_catboost_params = {
                           'learning_rate' : 0.923192518624813,
                           'max_depth' : 15,
                           'n_estimators' : 7748,
                           'reg_lambda' : 0.8598696247943665,
                           'subsample' : 0.9109367356405415,
                           'random_state' : 42
                            }

# Train model
print("Training CatBoost Tg model...")
catboost_Tg = XGBRegressor(**improved_catboost_params)
catboost_Tg.fit(
    X_train_split, y_train_split_Tg, 
    eval_set=[(X_val_split, y_val_split_Tg)],
    eval_metric = "mae",
    verbose = False
)

print("Training CatBoost FFV model...")
catboost_FFV = XGBRegressor(**improved_catboost_params)
catboost_FFV.fit(
    X_train_split, y_train_split_FFV, 
    eval_set=[(X_val_split, y_val_split_FFV)],
    eval_metric = "mae",
    verbose = False
)

print("Training CatBoost Tc model...")
catboost_Tc = CatBoostRegressor(**improved_catboost_params)
catboost_Tc.fit(
    X_train_split, y_train_split_Tc, 
    eval_set=[(X_val_split, y_val_split_Tc)],
    verbose = False
)

print("Training CatBoost Density model...")
catboost_Density = XGBRegressor(**improved_catboost_params)
catboost_Density.fit(
    X_train_split, y_train_split_Density, 
    eval_set=[(X_val_split, y_val_split_Density)],
    eval_metric = "mae",
    verbose = False
)

print("Training CatBoost Rg model...")
catboost_Rg = XGBRegressor(**improved_catboost_params)
catboost_Rg.fit(
    X_train_split, y_train_split_Rg, 
    eval_set=[(X_val_split, y_val_split_Rg)],
    eval_metric = "mae",
    verbose = False
)

Training CatBoost Tg model...
Training CatBoost FFV model...
Training CatBoost Tc model...
Training CatBoost Density model...
Training CatBoost Rg model...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.923192518624813,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=15, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=7748, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [16]:
# Make predictions
print("Making predictions...")

test_predictions_Tg = catboost_Tg.predict(X_test_processed)
test_predictions_FFV = catboost_FFV.predict(X_test_processed)
test_predictions_Tc = catboost_Tc.predict(X_test_processed)
test_predictions_Density = catboost_Density.predict(X_test_processed)
test_predictions_Rg = catboost_Rg.predict(X_test_processed)
print(test_predictions_Tg)

# Create submission
submission = pd.DataFrame({
    'id': test_final['id'],
    'Tg': test_predictions_Tg,
    'FFV': test_predictions_FFV, 
    'Tc': test_predictions_Tc,
    'Density': test_predictions_Density,
    'Rg': test_predictions_Rg
})

print("Feature importance (top 20):")
print(feature_importance.head(20))
print("Submission file saved as 'improved_submission.csv'")

Making predictions...
[ 6.0090706e+01  1.0660892e-04 -1.2438524e+02]
Feature importance (top 20):
             feature  importance
20      carbon_ratio    0.133485
0             length    0.106837
19  heteroatom_ratio    0.104265
18  complexity_score    0.083473
1          num_atoms    0.080464
6       carbon_count    0.075082
2          num_bonds    0.071188
3          num_rings    0.067556
4       num_branches    0.056821
7     nitrogen_count    0.050060
8       oxygen_count    0.044347
17      ether_groups    0.035306
5       num_aromatic    0.031755
9       sulfur_count    0.029811
11    fluorine_count    0.015259
12    chlorine_count    0.007379
13     bromine_count    0.005144
10  phosphorus_count    0.001767
15   carboxyl_groups    0.000000
16      ester_groups    0.000000
Submission file saved as 'improved_submission.csv'


In [17]:
submission.to_csv("submission.csv", index=False)
submission

,id,Tg,FFV,Tc,Density,Rg
0,1109053969,60.090706,0.322395,0.002924,0.002385,1.851558
1,1422188626,0.000107,0.381775,0.002444,0.000054,-0.000471
2,2032016830,-124.385239,0.081084,-0.005733,0.004167,0.070094
